# Lesson 01: Linear Regression with PyTorch

This notebook covers the fundamentals of PyTorch including:
- Tensor operations
- Building a neural network model
- Training loop
- Model evaluation and saving

## Part 1: Introduction to PyTorch Tensors

In [ ]:
# Import PyTorch library - the main deep learning framework we'll use
import torch

In [ ]:
# Create a tensor with random values
# torch.rand() creates a tensor filled with random numbers from a uniform distribution [0, 1)
# Shape: 3 rows x 4 columns
some_tensor = torch.rand(3, 4)
some_tensor

In [ ]:
# Display tensor properties
print(some_tensor)
# dtype: data type of tensor elements (float32 is default for torch.rand)
print(f"Datatype of tensor: {some_tensor.dtype}")
# shape: dimensions of the tensor (3x4 matrix)
print(f"Shape of tensor: {some_tensor.shape}")
# device: where the tensor is stored (CPU or CUDA GPU)
print(f"Device the tensor is on: {some_tensor.device}")

## Tensor Operations

In [ ]:
## Operation of tensors

In [ ]:
# Create a 3x3 tensor with random values for demonstrations
tensor_1 = torch.rand(3, 3)
tensor_1   

In [ ]:
# Element-wise addition: adds 10 to every element in the tensor
# This is broadcasting - the scalar 10 is "broadcast" to match tensor shape
tensor_1 + 10

In [ ]:
# Element-wise multiplication: multiplies every element by 10
tensor_1 * 10

In [ ]:
# Alternative way to perform element-wise multiplication using torch function
# torch.mul() is equivalent to the * operator
torch.mul(tensor_1, 8)

In [ ]:
# Alternative way to perform element-wise addition using torch function
# torch.add() is equivalent to the + operator
torch.add(tensor_1, 9)

In [ ]:
# Create a 1D tensor (vector) with specific values
tensor = torch.tensor([1, 2, 3])

In [ ]:
# %%time is a Jupyter magic command that measures execution time
# Matrix multiplication (dot product) of tensor with itself
# [1,2,3] · [1,2,3] = 1*1 + 2*2 + 3*3 = 14
%%time
torch.matmul(tensor, tensor)

## Tensor Manipulation: Reshaping and Stacking

In [ ]:
# Create a tensor with values from 2 to 98 (exclusive) with step of 2
# Result: [2, 4, 6, 8, ..., 96, 98] - 49 elements total
x = torch.arange(2, 99, 2)

In [ ]:
# Check the shape (number of elements) in the tensor
# Expected: torch.Size([49]) - a 1D tensor with 49 elements
x.shape

In [ ]:
# Reshape the 1D tensor into a 7x7 2D tensor (matrix)
# Note: total elements must match (49 = 7*7)
x_reshaped = x.reshape(7, 7)

In [ ]:
# Display the reshaped 7x7 matrix
x_reshaped

In [ ]:
# Create two 1D tensors with random values (3 elements each)
a = torch.rand(3)
b = torch.rand(3)
a, b

In [ ]:
# Stack tensors along dimension 1 (columns)
# Combines [a] and [b] into a 3x2 matrix where a and b are columns
# dim=0 would stack them as rows instead
torch.stack([a, b], dim=1)

In [ ]:
# squeeze() removes dimensions of size 1
# Note: x_reshaped is 7x7, so squeeze(1) doesn't change it (dimension 1 is not size 1)
x_reshaped.squeeze(1)

In [ ]:
# Create a 4D tensor filled with zeros
# Shape: 2 x 3 x 1 x 6 (the dimension of size 1 can be squeezed out)
zeros = torch.zeros(2, 3, 1, 6)
zeros

In [ ]:
# Import NumPy for array operations
import numpy as np

In [ ]:
# squeeze() removes ALL dimensions of size 1
# Original shape: (2, 3, 1, 6) -> After squeeze: (2, 3, 6)
np.shape(zeros.squeeze())

In [ ]:
# unsqueeze(dim=1) adds a new dimension of size 1 at position 1
# Original: (2, 3, 1, 6) -> After unsqueeze: (2, 1, 3, 1, 6)
np.shape(zeros.unsqueeze(dim=1))

## GPU/CUDA Availability Check

In [ ]:
# Check if CUDA (NVIDIA GPU support) is available on this system
# Returns True if GPU is available, False otherwise
torch.cuda.is_available()

In [ ]:
# Set device to GPU if available, otherwise use CPU
# This is a common pattern for writing device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Display which device will be used for computations
device

In [ ]:
# Count how many CUDA-capable GPUs are available
# Returns 0 if no GPUs, or number of available GPUs
torch.cuda.device_count()

## Part 2: Linear Regression - Creating Training Data

In [ ]:
# Import necessary libraries
import torch
from torch import nn  # nn contains neural network building blocks
import matplotlib.pyplot as plt  # for plotting
import numpy as np

# Display PyTorch version (useful for debugging compatibility issues)
torch.__version__

In [ ]:
# Create parameters for our synthetic linear data
# We'll create data following the equation: y = weight * x + bias
weight = 0.7
bias = 0.3

# Create input data (features)
# Generate values from 0 to 1 with step 0.02 (50 points)
start = 0
end = 1
step = 0.02
X = torch.arange(start, end, step).unsqueeze(dim=1)  # unsqueeze adds a dimension to make it a column vector

# Create output data (labels) using linear relationship
# This is our "ground truth" that the model will try to learn
y = weight * X + bias

In [ ]:
# Inspect the first 10 data points and their shapes
# X: input features, y: target values
# Both should be column vectors with shape (50, 1)
X[:10], y[:10], np.shape(X), np.shape(y)

## Train-Test Split

In [ ]:
# Create a train-test split (80% train, 20% test)
# This helps us evaluate how well the model generalizes to unseen data
train_split = int(0.8 * len(X))  # Calculate index for 80% of data
train_split

In [ ]:
# Split data into training and testing sets
# Training set: used to train the model (first 80% of data)
X_train, y_train = X[:train_split], y[:train_split]

# Test set: used to evaluate the model (last 20% of data)
X_test, y_test = X[train_split:], y[train_split:]

## Visualization Function

In [ ]:
# Define a helper function to visualize our data and predictions
def plot_predictions(
        train_data=X_train,
        train_labels=y_train,
        test_data=X_test,
        test_labels=y_test,
        predictions=None):
    """
    Plots training data, test data, and model predictions.
    
    Args:
        train_data: Input features for training
        train_labels: Target values for training
        test_data: Input features for testing
        test_labels: Target values for testing
        predictions: Model predictions (optional)
    """
    
    # Create a figure with specified size
    plt.figure(figsize=(10, 7))
    
    # Plot training data in one color
    plt.scatter(train_data, train_labels, label="Train data")
    
    # Plot test data in another color
    plt.scatter(test_data, test_labels, label="Test data")

    # If predictions exist, plot them to compare with actual test data
    if predictions is not None:
        plt.scatter(test_data, predictions, label="Predictions")

    # Add legend with larger font
    plt.legend(prop={"size": 14})
    plt.show()

In [ ]:
# Visualize the train-test split
# Should show training points (blue) and test points (orange) in a linear pattern
plot_predictions()

## Part 3: Building the Linear Regression Model

In [ ]:
# Create a Linear Regression model class
# This implements the equation: y = weight * x + bias
class LinearRegressionModel(nn.Module):  # Inherit from nn.Module (base class for all neural networks)
    def __init__(self):
        """
        Initialize the model's parameters (weights and bias).
        These are the values the model will learn during training.
        """
        super().__init__()  # Call parent class constructor
        
        # Initialize weights as a learnable parameter
        # nn.Parameter tells PyTorch this value should be optimized during training
        self.weights = nn.Parameter(
            torch.randn(1, dtype=torch.float),  # Start with random value (will be adjusted)
            requires_grad=True  # Enable gradient computation for this parameter
        )

        # Initialize bias as a learnable parameter
        self.bias = nn.Parameter(
            torch.randn(1, dtype=torch.float),  # Start with random value (will be adjusted)
            requires_grad=True  # Enable gradient computation for this parameter
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass: defines how input data flows through the model.
        
        Args:
            x: Input features (training or testing data)
            
        Returns:
            Predictions based on current weights and bias
        """
        # Linear regression formula: y = mx + b
        return self.weights * x + self.bias

In [ ]:
# Set manual seed for reproducibility
# This ensures we get the same random initialization every time we run the code
torch.manual_seed(42)

# Create an instance of our model
# This creates a specific LinearRegressionModel object with randomly initialized parameters
model_0 = LinearRegressionModel()

# Display the model's parameters (weights and bias)
# These are the values that will be learned during training
list(model_0.parameters())

In [ ]:
# Display parameters with their names in a dictionary format
# state_dict() is commonly used for saving/loading models
model_0.state_dict()

## Making Predictions (Before Training)

In [ ]:
# Make predictions with the untrained model
# torch.inference_mode() disables gradient tracking (faster, uses less memory)
# Used during evaluation/testing, not training
with torch.inference_mode(): 
    y_preds = model_0(X_test)  # Pass test data through the model

# Note: In older PyTorch code you might see torch.no_grad() instead
# torch.inference_mode() is newer and more efficient
# with torch.no_grad():
#     y_preds = model_0(X_test)

In [ ]:
# Check the predictions from the untrained model
# These will be poor because the model hasn't learned anything yet
print(f"Number of testing samples: {len(X_test)}") 
print(f"Number of predictions made: {len(y_preds)}")
print(f"Predicted values:\n{y_preds}")

In [ ]:
# Visualize predictions from untrained model
# The predictions (green) should be very different from the actual test data (orange)
plot_predictions(predictions=y_preds)

## Part 4: Training the Model

In [ ]:
# Create the loss function
# L1Loss (Mean Absolute Error) measures average absolute difference between predictions and actual values
# Formula: MAE = (1/n) * Σ|predicted - actual|
loss_fn = nn.L1Loss()

# Create the optimizer
# SGD (Stochastic Gradient Descent) updates parameters to minimize loss
optimizer = torch.optim.SGD(
    params=model_0.parameters(),  # Parameters to optimize (weights and bias)
    lr=0.01  # Learning rate: how big each update step is (higher = faster but less stable)
)

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Set the number of epochs (complete passes through the training data)
# More epochs = more learning opportunities, but can lead to overfitting
epochs = 200

# Create empty lists to track loss values over time
# Useful for visualizing training progress
epoch_count = []
loss_values = []
test_loss_values = []

# Training loop
for epoch in range(epochs):
    ### Training Phase ###

    # Put model in training mode
    # This enables gradient tracking and dropout/batch norm (if used)
    model_0.train()

    # 1. Forward pass: compute predictions using current parameters
    y_pred = model_0(X_train)

    # 2. Calculate loss: how wrong are our predictions?
    # Lower loss = better predictions
    loss = loss_fn(y_pred, y_train)

    # 3. Zero gradients from previous iteration
    # Gradients accumulate by default, so we need to reset them
    optimizer.zero_grad()

    # 4. Backpropagation: compute gradients of loss with respect to parameters
    # This calculates how much each parameter contributed to the error
    loss.backward()

    # 5. Update parameters using gradients
    # Optimizer adjusts weights and bias to reduce loss
    optimizer.step()

    ### Testing Phase ###

    # Put model in evaluation mode
    # This disables dropout/batch norm and other training-specific behaviors
    model_0.eval()

    # Disable gradient tracking for evaluation (saves memory and computation)
    with torch.inference_mode():
        # 1. Forward pass on test data
        test_pred = model_0(X_test)

        # 2. Calculate test loss to see how well model generalizes
        test_loss = loss_fn(test_pred, y_test)

    # Print progress every 10 epochs
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss} | Test loss: {test_loss}")

        # Store values for plotting
        epoch_count.append(epoch)
        loss_values.append(loss)
        test_loss_values.append(test_loss)

In [ ]:
# Convert lists to NumPy arrays for easier manipulation
epoch_count = np.array(epoch_count)
test_loss_values = np.array(test_loss_values)

## Visualizing Training Progress

In [ ]:
# Plot how loss decreased during training
# This helps us understand if the model is learning properly
plt.scatter(epoch_count, torch.tensor(loss_values).numpy(), label="Train loss")
plt.scatter(epoch_count, test_loss_values, label="Test loss")

# Use logarithmic scale on y-axis for better visualization
# Loss values can span several orders of magnitude
plt.yscale("log")

plt.xlabel("Epoches")
plt.ylabel("Loss")
plt.legend()
plt.show()

## Evaluating the Trained Model

In [ ]:
# Display the learned parameters
# These should be close to our original values: weight=0.7, bias=0.3
model_0.state_dict()

In [ ]:
# Show the true parameters we used to generate the data
# Compare these with the learned parameters above
weight, bias

In [ ]:
# Make predictions with the trained model
with torch.inference_mode(): 
    y_preds = model_0(X_test)

In [ ]:
# Visualize predictions from trained model
# Now predictions (green) should closely match test data (orange)
plot_predictions(predictions=y_preds)

## Part 5: Saving and Loading the Model

In [ ]:
# Import Path for handling file paths in a cross-platform way
from pathlib import Path

In [ ]:
# 1. Create a directory to store models
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)  # parents=True creates parent dirs, exist_ok=True doesn't error if exists

# 2. Create the full path for saving the model
MODEL_NAME = "01_pytorch_workflow_model_0.pth"  # .pth is PyTorch convention
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# 3. Save the model's state dictionary (learned parameters)
print(f"Saving the model to {MODEL_SAVE_PATH}")
torch.save(
    obj=model_0.state_dict(),  # Save only the parameters, not the entire model structure
    f=MODEL_SAVE_PATH
)

In [ ]:
# Verify the saved parameters
model_0.state_dict()

In [ ]:
# Load the saved model
# Step 1: Create a new instance of the model class (with random initialization)
loaded_model_0 = LinearRegressionModel()

# Step 2: Load the saved parameters into the new model
# This replaces the random parameters with our trained values
loaded_model_0.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

In [ ]:
# Verify the loaded parameters match the original model
loaded_model_0.state_dict()

In [ ]:
# Make predictions with the loaded model
# Set to evaluation mode first
loaded_model_0.eval()

with torch.inference_mode():
    loaded_model_preds = loaded_model_0(X_test)

loaded_model_preds

In [ ]:
# Make predictions with original model for comparison
y_pred = model_0(X_test)

In [ ]:
# Verify that loaded model produces identical predictions to original model
# All values should be True, confirming successful save/load
y_preds == loaded_model_preds